![Piksel Sandbox](../../assets/piksel_header.png)

# Xarray for ODC

## A. The xarray Dataset and DataArray

A satellite observation produces a grid of pixel values: one number per pixel, for one band, at one moment in time.
A typical Earth observation query covers several bands and multiple dates over the same area, so the result is many grids of that same shape, each tied to a particular band and date.

A plain numerical array stores only the pixel values, leaving the dates, pixel positions, projection, and band names to be tracked separately.
xarray attaches these labels to the values, so they travel together through every operation.

Two types do the work.

- **DataArray** is one named array of pixel values, labelled along each of its dimensions.
  The DataArrays returned by `dc.load` have three dimensions (`time`, `y`, `x`) and a name such as `red`.
  Indexing a DataArray, doing arithmetic on it, or saving it always keeps these labels attached.

- **Dataset** is a collection of DataArrays that share the same dimensions and coordinates.
  Each measurement returned by `dc.load` (`red`, `green`, `blue`, `nir`) appears as one DataArray inside the Dataset, and they all line up on the same `time`, `y`, `x` grid.
  A Dataset can also carry notes that apply to all its arrays, such as the map projection.

![One DataArray drawn as a single labelled grid over y/latitude and x/longitude, alongside one Dataset drawn as a matrix of four bands (red, green, blue, nir) across four dates, with all bands sharing the same y, x grid.](../../assets/xarray-dataarray-vs-dataset.svg)

`dc.load` always returns its result as a Dataset.
Section D illustrates this structure against the dataset loaded in section C.
The xarray documentation gives the formal definitions: [xarray data structures](https://docs.xarray.dev/en/stable/user-guide/data-structures.html).

## B. Outline

1. Load a small sample dataset using the pattern from notebook 03.
2. Inspect its dimensions, coordinates, and data variables.
3. Select subsets along a dimension with `.sel` and `.isel`.
4. Combine variables with arithmetic.

## C. Loading a sample dataset

The load call reuses the area, grid, and product from notebook 03 over a two-year range.
The returned `ds` is an `xarray.Dataset`; its representation below shows the dimensions, coordinates, and data variables that make up its structure.

In [ ]:
import datacube

dc = datacube.Datacube(app="xarray_for_odc")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2024", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)
ds

## D. Reading the Dataset

The Dataset printed in section C is organised into four parts: dimensions, data variables, coordinates, and attributes.

![Structure of an xarray.Dataset returned by dc.load, showing the four parts: dimensions, data variables, coordinates, and attributes.](../../assets/xarray-dataset-structure.svg)

1. **Dimensions** record the size of the data.

In [ ]:
ds.sizes

Here, two time slices and a 369 × 372 grid of pixels at 30 m resolution. This sizes quickly informed us before running any computational against a dataset. In this case, with 2 x 369 x 372 dimension, the computation should be quite light-weight.

2. **Data variables** record what was measured.

In [ ]:
ds.data_vars

Here, four surface-reflectance bands: red, green, blue, and nir. This also give a quick overview of the data sizes of 549kB and the band value, in some case just looking at the number would tell us directly about the data. For example, a surface temperature band, looking at this we might understand if the data is still in digital number or already in temperature degree.

To see more detail in a specific band:

In [ ]:
ds.data_vars["red"]

3. **Coordinates** label every slice and every pixel values.

In [ ]:
ds.coords

This is good to know, despite we rarely use it. 

4. **Attributes** record facts that apply to the whole Dataset.

In [ ]:
ds.attrs

The CRS is recorded as EPSG:32647 (Identification code for the WGS 84 / UTM zone 47N projected coordinate system). Given this information, y and x can be read as metres without further reprojection.

## E. Selecting subsets of the Dataset

Selection pulls a piece of the Dataset out for further work.
The original `ds` is left unchanged.

Both `.sel` and `.isel` are methods on every xarray Dataset and DataArray.
A method is a function that belongs to an object and is called with dot notation.
A typical call looks like `ds.sel(time="2024")` or `ds.isel(time=0)`.
Inside the parentheses, `dimension=value` says which dimension to select on and which value to keep.

### Dimension lookup by label

`.sel` selects by coordinate label.
A year string, a date, a list of dates, or a slice of dates are all valid for `time`.

In [ ]:
ds.sel(time="2024")

The result is a new Dataset with only the 2024 time slice.
The y and x grids are unchanged.

Other selection patterns follow the same shape:

- a list of labels: `ds.sel(time=["2024-01-01", "2025-01-01"])`
- a range slice: `ds.sel(time=slice("2024", "2025"))`
- a boolean mask: `ds.sel(time=ds.time.dt.year == 2024)`

### Dimension lookup by index

`.isel` selects by integer position, the same convention as NumPy.

In [ ]:
ds.isel(time=0)

This returns the first time slice by its index rather than by its date.
A list of indices (`ds.isel(time=[0, 1])`) or a slice (`ds.isel(time=slice(0, 2))`) selects multiple positions.
`.isel` is useful when the labels are not known or not convenient to type.

## F. Arithmetic across variables

Each measurement in the Dataset is a DataArray, accessed by name: `ds.red`, `ds.nir`, and so on.
Two DataArrays that share the same dimensions and coordinates can be combined with the usual arithmetic operators.

The **Normalised Difference Vegetation Index (NDVI)** is a classic example.
It contrasts near-infrared reflectance against red reflectance to estimate vegetation density:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

Healthy vegetation reflects strongly in the near-infrared and absorbs red, so NDVI is high over forests and crops, low over bare ground, and negative over water.

In [ ]:
nir = ds.nir.astype("float64")
red = ds.red.astype("float64")

ndvi = (nir - red) / (nir + red)
ndvi

The subtraction, addition, and division each run element-wise across every pixel and every time slice.
The result is a new DataArray on the same `time`, `y`, and `x` dimensions, with values in the range [-1, 1].

The bands are cast to `float32` before the arithmetic.
Data variables come from `dc.load` as `uint16` (integer-encoded reflectance), and integer subtraction underflows when red exceeds nir at a pixel.
Casting to a floating-point type keeps the arithmetic well-defined.

## G. Next steps

Notebook 05 uses these structures to draw true-colour composites and single-band images: [`05_plotting.ipynb`](./05_plotting.ipynb).